## DBLP-Quad Update
The purpose of this notebook is to update the queries to use IRIs for venues and run the queries from the dblp-Quad dataset to get the current results.
The results from the dataset might be deprecated.

#### Edit in the Venue IRIs

In [1]:
import os, sys
# optional: make notebook treat repo root as working dir
os.chdir('..')
sys.path.insert(0, os.path.abspath('backend'))

In [2]:
# Jupyter magic to display all columns of a dataframe
import pandas as pd
from IPython.display import display
pd.set_option('display.max_columns', None)  # Show all columns in the dataframe
pd.set_option('display.max_rows', 10)     # Show all rows in the dataframe (

In [3]:
import json

train_questions_path = "./backend/evaluation/data/dblp-quad/train.questions.json"
train_answers = "./backend/evaluation/data/dblp-quad/train.answers.json"

with open(train_questions_path, encoding='utf-8') as f:
    raw_data = json.load(f)
    train_questions = pd.DataFrame.from_records(raw_data["questions"])

In [4]:
# train_questions = train_questions[train_questions["query"].apply(lambda q: q['sparql'].startswith("SELECT"))] # Filter query type for SELECT
train_questions.reset_index(drop=True, inplace=True)
print("Total questions:", len(train_questions))

#Remove query types that are not allowed for evaluation
allowed_query_types = ["SINGLE_FACT", "MULTI_FACT", "DOUBLE_INTENT", "BOOLEAN", "NEGATION"]

train_questions = train_questions[train_questions["query_type"].isin(allowed_query_types)]
print("Questions after filtering by allowed query types:", len(train_questions))

Total questions: 7000
Questions after filtering by allowed query types: 3500


In [5]:
# Ensure query_type is a column and not the index
if "query_type" not in train_questions.columns:
    train_questions = train_questions.reset_index()

# Now group and take the top 20
train_questions_limited = train_questions.groupby("query_type").head(20).reset_index(drop=True)

# print out the count of each query type in the limited dataframe
print("\nCount of each query type in the limited dataframe:")
print(train_questions_limited["query_type"].value_counts())


Count of each query type in the limited dataframe:
query_type
SINGLE_FACT      20
MULTI_FACT       20
DOUBLE_INTENT    20
BOOLEAN          20
NEGATION         20
Name: count, dtype: int64


In [6]:
# print all the unique relations in the dataset
unique_relations = set(rel for rels in train_questions_limited["relations"] for rel in rels)
unique_relations

{'<https://dblp.org/rdf/schema#authoredBy>',
 '<https://dblp.org/rdf/schema#primaryAffiliation>',
 '<https://dblp.org/rdf/schema#publishedIn>',
 '<https://dblp.org/rdf/schema#title>',
 '<https://dblp.org/rdf/schema#webpage>',
 '<https://dblp.org/rdf/schema#wikidata>',
 '<https://dblp.org/rdf/schema#yearOfPublication>'}

In [8]:
def filter_relation(questions_df, relation):
    return questions_df[questions_df["relations"].apply(lambda rels: relation in rels)]

publishedIn_relation = "<https://dblp.org/rdf/schema#publishedIn>"
publishedIn_questions = filter_relation(train_questions_limited, publishedIn_relation)
publishedIn_questions.count()

id                      36
query_type              36
question                36
paraphrased_question    36
query                   36
template_id             36
entities                36
relations               36
temporal                36
held_out                36
dtype: int64

In [9]:
# get all entities (enclosed in '') after <https://dblp.org/rdf/schema#publishedIn> in query: sparql: and print them out
publishedIn_entities = set()
for query in publishedIn_questions["query"]:
    if "sparql" in query:
        sparql_query = query["sparql"]
        # Find the part of the query after the relation
        parts = sparql_query.split(publishedIn_relation)
        if len(parts) > 1:
            # Get the part after the relation and extract entities enclosed in ''
            after_relation = parts[1]
            entities = [part.split("'")[1] for part in after_relation.split() if part.startswith("'") and part.endswith("'")]
            publishedIn_entities.update(entities)

print("Number of unique entities related to publishedIn relation:", len(publishedIn_entities))
print("Entities related to publishedIn relation:")
for entity in publishedIn_entities:
    print(entity)

Number of unique entities related to publishedIn relation: 9
Entities related to publishedIn relation:
ICCD
ICMV
NetSys
CoRR
CAMP
CF
ACSSC
ICRAI
MMAR


In [10]:
from httpx import AsyncClient
import asyncio

DBLP_VENUE_API= "https://dblp.org/search/venue/api"


async def link_venue_to_dblp(venue_name: str, max_results: int = 20):
    params = {"q": venue_name, "format": "json", "h": max_results}

    async with AsyncClient(
        timeout=10.0,
        headers={"User-Agent": "DBLP-Quad-Linker/1.0 (Julius Kaltwasser, RWTH Aachen University)"}
    ) as client:
        resp = await client.get(DBLP_VENUE_API, params=params)
    resp.raise_for_status()
    data = resp.json()
    hits = data.get("result", {}).get("hits", {}).get("hit", [])
    candidates = []
    for h in hits:
        info = h.get("info", {})
        # Extract the venue key from the URL (e.g., https://dblp.org/db/conf/nips/ -> conf/nips)
        url = info.get("url", "")
        key = url.replace("https://dblp.org/db/", "").rstrip("/") if url else None
        candidates.append({
            "name":    info.get("venue"),
            "url":     url,
            "dblp_id": key,
            "hit_id":  h.get("@id"),  # Also store the hit ID
            "type": "Venue"
        })
    return candidates


In [11]:
venues = list(publishedIn_entities)

results = []

for venue in venues:
    candidates = await link_venue_to_dblp(venue)
    results.append(candidates)
    print(f"Queried DBLP for venue: {venue}, found {len(candidates)} candidates.")
    await asyncio.sleep(1)

# Process results
for venue, candidates in zip(venues, results):
    print(f"\nLinking venue: {venue}")
    for c in candidates:
        print(c)

Queried DBLP for venue: ICCD, found 3 candidates.
Queried DBLP for venue: ICMV, found 2 candidates.
Queried DBLP for venue: NetSys, found 1 candidates.
Queried DBLP for venue: CoRR, found 7 candidates.
Queried DBLP for venue: CAMP, found 2 candidates.
Queried DBLP for venue: CF, found 7 candidates.
Queried DBLP for venue: ACSSC, found 1 candidates.
Queried DBLP for venue: ICRAI, found 1 candidates.
Queried DBLP for venue: MMAR, found 1 candidates.

Linking venue: ICCD
{'name': 'International Conference on Compute and Data Analysis (ICCDA)', 'url': 'https://dblp.org/db/conf/iccda/', 'dblp_id': 'conf/iccda', 'hit_id': '3433', 'type': 'Venue'}
{'name': 'International Conference on Computer Design (ICCD)', 'url': 'https://dblp.org/db/conf/iccd/', 'dblp_id': 'conf/iccd', 'hit_id': '3442', 'type': 'Venue'}
{'name': 'International Conferences on Computing and Data Engineering (ICCDE)', 'url': 'https://dblp.org/db/conf/iccde/', 'dblp_id': 'conf/iccde', 'hit_id': '4729', 'type': 'Venue'}

Linki

In [ ]:
# pick the best candidate for each venue and print it out
for venue, candidates in zip(venues, results):
    if candidates:
        best_candidate = candidates[0]  # Assuming the first candidate is the best one
        print(f"Best candidate for venue '{venue}': {best_candidate}")
    else:
        print(f"No candidates found for venue '{venue}'")

Best candidate for venue 'ICCD': {'name': 'International Conference on Compute and Data Analysis (ICCDA)', 'url': 'https://dblp.org/db/conf/iccda/', 'dblp_id': 'conf/iccda', 'hit_id': '3433', 'type': 'Venue'}
Best candidate for venue 'ICMV': {'name': 'International Conference on Machine Vision (ICMV)', 'url': 'https://dblp.org/db/conf/icmv/', 'dblp_id': 'conf/icmv', 'hit_id': '4143', 'type': 'Venue'}
Best candidate for venue 'NetSys': {'name': 'International Conference on Networked Systems (NetSys)', 'url': 'https://dblp.org/db/conf/kivs/', 'dblp_id': 'conf/kivs', 'hit_id': '4256', 'type': 'Venue'}
Best candidate for venue 'CoRR': {'name': 'ACM-SIGPLAN Symposium on Programming Language Design and Implementation (PLDI)', 'url': 'https://dblp.org/db/conf/pldi/', 'dblp_id': 'conf/pldi', 'hit_id': '232', 'type': 'Venue'}
Best candidate for venue 'CAMP': {'name': 'Information Retrieval &amp; Knowledge Management (CAMP)', 'url': 'https://dblp.org/db/conf/infrkm/', 'dblp_id': 'conf/infrkm', '

In [13]:
# manual matching for venues that were not correctly linked
# for ICCD pick second candidate
# for CoRR pick second candidate
# rest keeps first candidate
final_links = {}
for venue, candidates in zip(venues, results):
    if candidates:
        if venue == "ICCD":
            final_links[venue] = candidates[1] if len(candidates) > 1 else candidates[0]
        elif venue == "CoRR":
            final_links[venue] = candidates[1] if len(candidates) > 1 else candidates[0]
        else:
            final_links[venue] = candidates[0]
    else:
        final_links[venue] = None

print("\nFinal linked venues:")
for venue, link in final_links.items():
    print(f"{venue}: {link}")


Final linked venues:
ICCD: {'name': 'International Conference on Computer Design (ICCD)', 'url': 'https://dblp.org/db/conf/iccd/', 'dblp_id': 'conf/iccd', 'hit_id': '3442', 'type': 'Venue'}
ICMV: {'name': 'International Conference on Machine Vision (ICMV)', 'url': 'https://dblp.org/db/conf/icmv/', 'dblp_id': 'conf/icmv', 'hit_id': '4143', 'type': 'Venue'}
NetSys: {'name': 'International Conference on Networked Systems (NetSys)', 'url': 'https://dblp.org/db/conf/kivs/', 'dblp_id': 'conf/kivs', 'hit_id': '4256', 'type': 'Venue'}
CoRR: {'name': 'Computing Research Repository (CoRR)', 'url': 'https://dblp.org/db/journals/corr/', 'dblp_id': 'journals/corr', 'hit_id': '1081', 'type': 'Venue'}
CAMP: {'name': 'Information Retrieval &amp; Knowledge Management (CAMP)', 'url': 'https://dblp.org/db/conf/infrkm/', 'dblp_id': 'conf/infrkm', 'hit_id': '2870', 'type': 'Venue'}
CF: {'name': 'ACM International Conference on Computing Frontiers (CF)', 'url': 'https://dblp.org/db/conf/cf/', 'dblp_id': 'c

In [ ]:
# Replace the venue names in the original sparql queries with the linked DBLP IDs and add the IRIs to entities


In [14]:
train_questions_limited.head()

,id,query_type,question,paraphrased_question,query,template_id,entities,relations,temporal,held_out
0,Q0001,SINGLE_FACT,{'string': 'What are the papers written by the...,{'string': 'Which papers did the author Wazir ...,{'sparql': 'SELECT DISTINCT ?answer WHERE { ?a...,TC01,[<https://dblp.org/pid/211/3355>],[<https://dblp.org/rdf/schema#authoredBy>],False,False
1,Q0002,SINGLE_FACT,{'string': 'What is the Wikidata ID of Yvo Des...,{'string': 'The author Y. Desmedt is associate...,{'sparql': 'SELECT DISTINCT ?answer WHERE { <h...,TC04,[<https://dblp.org/pid/d/YvoDesmedt>],[<https://dblp.org/rdf/schema#wikidata>],False,False
2,Q0003,SINGLE_FACT,{'string': 'What is the primary affiliation of...,{'string': 'Leandro Krug Wives is primarily af...,{'sparql': 'SELECT DISTINCT ?answer WHERE { <h...,TC02,[<https://dblp.org/pid/w/LeandroKrugWives>],[<https://dblp.org/rdf/schema#primaryAffiliati...,False,False
3,Q0004,SINGLE_FACT,{'string': 'What is the webpage of Ravi Kumar?'},{'string': 'What is the webpage of the person ...,{'sparql': 'SELECT DISTINCT ?answer WHERE { <h...,TC05,[<https://dblp.org/pid/k/RaviKumar>],[<https://dblp.org/rdf/schema#webpage>],False,False
4,Q0005,SINGLE_FACT,"{'string': 'Which publications did Lambe, Larr...","{'string': 'Which papers did the author Lambe,...",{'sparql': 'SELECT DISTINCT ?answer WHERE { ?a...,TC01,[<https://dblp.org/pid/13/378>],[<https://dblp.org/rdf/schema#authoredBy>],False,False


In [ ]:
import copy

train_questions_final = train_questions_limited.copy()
train_questions_final['query'] = train_questions_final['query'].apply(copy.deepcopy)

venue_to_iri = {name: f"<{info['url']}>" for name, info in final_links.items() if info}
pub_rel = "<https://dblp.org/rdf/schema#publishedIn>"

for index, row in train_questions_final.iterrows():
    if pub_rel in row['relations']:
        
        current_sparql = row['query']['sparql']
        current_entities = list(row['entities'])
        modified = False
        
        for venue_name, iri in venue_to_iri.items():
            literal_venue = f"'{venue_name}'"
            
            if literal_venue in current_sparql:
                current_sparql = current_sparql.replace(literal_venue, iri)

                if iri not in current_entities:
                    current_entities.append(iri)
                
                modified = True
    
        if modified:
            train_questions_final.at[index, 'query']['sparql'] = current_sparql
            train_questions_final.at[index, 'entities'] = current_entities

print(f"Total rows processed: {len(train_questions_final)}")

Total rows processed: 100

--- Comparison for Row ID: Q0352 ---
Original SPARQL: SELECT DISTINCT ?answer WHERE { ?x <https://dblp.org/rdf/schema#authoredBy> <https://dblp.org/pid/274/3007> . ?x <https://dblp.org/rdf/schema#publishedIn> ?answer }
Updated SPARQL:  SELECT DISTINCT ?answer WHERE { ?x <https://dblp.org/rdf/schema#authoredBy> <https://dblp.org/pid/274/3007> . ?x <https://dblp.org/rdf/schema#publishedIn> ?answer }
Updated Entities: ['<https://dblp.org/pid/274/3007>']


In [21]:
changed_rows = train_questions_final[train_questions_final['relations'].apply(lambda rels: pub_rel in rels)]
for qtype in changed_rows['query_type'].unique():
    sample_row = changed_rows[changed_rows['query_type'] == qtype].iloc[0]
    print(f"\n--- Sample for Query Type: {qtype} ---")
    print(f"ID: {sample_row['id']}")
    print(f"Original SPARQL: {train_questions_limited[train_questions_limited['id'] == sample_row['id']]['query'].iloc[0]['sparql']}")
    print(f"Updated SPARQL:  {sample_row['query']['sparql']}")
    print(f"Updated Entities: {sample_row['entities']}")


--- Sample for Query Type: MULTI_FACT ---
ID: Q0352
Original SPARQL: SELECT DISTINCT ?answer WHERE { ?x <https://dblp.org/rdf/schema#authoredBy> <https://dblp.org/pid/274/3007> . ?x <https://dblp.org/rdf/schema#publishedIn> ?answer }
Updated SPARQL:  SELECT DISTINCT ?answer WHERE { ?x <https://dblp.org/rdf/schema#authoredBy> <https://dblp.org/pid/274/3007> . ?x <https://dblp.org/rdf/schema#publishedIn> ?answer }
Updated Entities: ['<https://dblp.org/pid/274/3007>']

--- Sample for Query Type: DOUBLE_INTENT ---
ID: Q0702
Original SPARQL: SELECT DISTINCT ?firstanswer ?secondanswer WHERE { ?x <https://dblp.org/rdf/schema#authoredBy> <https://dblp.org/pid/m/HosamMMahmoud> . ?x <https://dblp.org/rdf/schema#publishedIn> ?firstanswer . ?x <https://dblp.org/rdf/schema#title> ?secondanswer }
Updated SPARQL:  SELECT DISTINCT ?firstanswer ?secondanswer WHERE { ?x <https://dblp.org/rdf/schema#authoredBy> <https://dblp.org/pid/m/HosamMMahmoud> . ?x <https://dblp.org/rdf/schema#publishedIn> ?firsta

In [ ]:
import json

# Define your output path
output_path = "./backend/evaluation/data/dblp-quad/subset/train.questions.updated.json"

# 1. Convert DataFrame back to a list of dictionaries
# 'records' ensures each row is a {column: value} object
questions_list = train_questions_final.to_dict(orient='records')

# 2. Wrap the list in a dictionary with the "questions" key
final_json_structure = {"questions": questions_list}

# 3. Save to file
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(final_json_structure, f, ensure_ascii=False, indent=2)

#### Fix the Year literals

#### Run the Queries on the DBLP endpoint

In [35]:
import json
import time
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON

def fetch_limited_sparql_results(df, output_filepath, endpoint_url, limit_per_type):
    if limit_per_type:
        # 1. Dynamically get N samples for EVERY unique type in the dataframe
        subset_df = (
            df.groupby('query_type', as_index=False)
            .head(limit_per_type)
        )
    else:
        subset_df = df.copy()

    # 2. Print Summary using value_counts (auto-detects all types)
    print("Detected Types and Counts to Process:")
    print(subset_df['query_type'].value_counts().to_string())
    print("-" * 30)

    # 3. Setup SPARQL
    sparql = SPARQLWrapper(endpoint_url)
    sparql.setReturnFormat(JSON)
    output_results = {"answers": []}

    print(f"Starting API calls for {len(subset_df)} total items...\n")

    # 4. Iterate over the refined DataFrame
    for _, row in subset_df.iterrows():
        entry_id = row['id']
        q_type = row['query_type']
        
        # Safely extract the sparql string
        query_data = row.get('query')
        query_string = query_data.get('sparql') if isinstance(query_data, dict) else None

        if not query_string:
            print(f"  [SKIP] No SPARQL query found for {entry_id}")
            continue

        try:
            sparql.setQuery(query_string)
            response = sparql.query().convert()

            # ASK queries return a 'boolean' key, SELECT queries return 'results'
            is_boolean = "boolean" in response
            
            result_item = {
                "id": entry_id,
                "query_type": q_type,
                "answer": {
                    "head": response.get("head", {}),
                    "results": response.get("results", {}) if not is_boolean else {},
                    "boolean": response.get("boolean") if is_boolean else None
                }
            }
            output_results["answers"].append(result_item)
            print(f"  [OK] Processed {entry_id} ({q_type})")

            # Respectful delay for the endpoint
            time.sleep(1)

        except Exception as e:
            print(f"  [ERROR] Failed {entry_id}: {str(e)}")

    # 5. Save results
    with open(output_filepath, 'w', encoding='utf-8') as f:
        json.dump(output_results, f, indent=4)

    print(f"\nDone! Results saved to {output_filepath}")

In [29]:
train_questions_final.head()

,id,query_type,question,paraphrased_question,query,template_id,entities,relations,temporal,held_out
0,Q0001,SINGLE_FACT,{'string': 'What are the papers written by the...,{'string': 'Which papers did the author Wazir ...,{'sparql': 'SELECT DISTINCT ?answer WHERE { ?a...,TC01,[<https://dblp.org/pid/211/3355>],[<https://dblp.org/rdf/schema#authoredBy>],False,False
1,Q0002,SINGLE_FACT,{'string': 'What is the Wikidata ID of Yvo Des...,{'string': 'The author Y. Desmedt is associate...,{'sparql': 'SELECT DISTINCT ?answer WHERE { <h...,TC04,[<https://dblp.org/pid/d/YvoDesmedt>],[<https://dblp.org/rdf/schema#wikidata>],False,False
2,Q0003,SINGLE_FACT,{'string': 'What is the primary affiliation of...,{'string': 'Leandro Krug Wives is primarily af...,{'sparql': 'SELECT DISTINCT ?answer WHERE { <h...,TC02,[<https://dblp.org/pid/w/LeandroKrugWives>],[<https://dblp.org/rdf/schema#primaryAffiliati...,False,False
3,Q0004,SINGLE_FACT,{'string': 'What is the webpage of Ravi Kumar?'},{'string': 'What is the webpage of the person ...,{'sparql': 'SELECT DISTINCT ?answer WHERE { <h...,TC05,[<https://dblp.org/pid/k/RaviKumar>],[<https://dblp.org/rdf/schema#webpage>],False,False
4,Q0005,SINGLE_FACT,"{'string': 'Which publications did Lambe, Larr...","{'string': 'Which papers did the author Lambe,...",{'sparql': 'SELECT DISTINCT ?answer WHERE { ?a...,TC01,[<https://dblp.org/pid/13/378>],[<https://dblp.org/rdf/schema#authoredBy>],False,False


In [36]:
# Pass the DataFrame instead of the file path
answers_output_path = "./backend/evaluation/data/dblp-quad/subset/train.answers.processed.json"
fetch_limited_sparql_results(train_questions_final, answers_output_path, "https://sparql.dblp.org/sparql", limit_per_type=5)

Detected Types and Counts to Process:
query_type
SINGLE_FACT      5
MULTI_FACT       5
DOUBLE_INTENT    5
BOOLEAN          5
NEGATION         5
------------------------------
Starting API calls for 25 total items...

  [OK] Processed Q0001 (SINGLE_FACT)
  [OK] Processed Q0002 (SINGLE_FACT)
  [OK] Processed Q0003 (SINGLE_FACT)
  [OK] Processed Q0004 (SINGLE_FACT)
  [OK] Processed Q0005 (SINGLE_FACT)
  [OK] Processed Q0351 (MULTI_FACT)
  [OK] Processed Q0352 (MULTI_FACT)
  [OK] Processed Q0353 (MULTI_FACT)
  [OK] Processed Q0354 (MULTI_FACT)
  [OK] Processed Q0355 (MULTI_FACT)
  [OK] Processed Q0701 (DOUBLE_INTENT)
  [OK] Processed Q0702 (DOUBLE_INTENT)
  [OK] Processed Q0703 (DOUBLE_INTENT)
  [OK] Processed Q0704 (DOUBLE_INTENT)
  [OK] Processed Q0705 (DOUBLE_INTENT)
  [OK] Processed Q1051 (BOOLEAN)
  [OK] Processed Q1052 (BOOLEAN)
  [OK] Processed Q1053 (BOOLEAN)
  [OK] Processed Q1054 (BOOLEAN)
  [OK] Processed Q1055 (BOOLEAN)
  [ERROR] Failed Q1401: EndPointInternalError: The endpoin